In [43]:
import pandas as pd

In [44]:
df=pd.read_csv("/content/sample_data/labeled_data.csv")
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,review
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [45]:
df.shape

(24783, 7)

In [46]:
df.isna().sum()

,0
Unnamed: 0,0
count,0
hate_speech,0
offensive_language,0
neither,0
class,0
review,0


In [47]:
df.drop_duplicates(inplace=True)

df.shape

(24783, 7)

In [48]:
df["review"] = df["review"].astype(str)

# text preprocessing

In [49]:
import re
## 1. to lower case
df["review"]=df["review"].str.lower()

## 2 remove URLs
def remove_url(text):
    return re.sub(r"http\S+", "", text) # (pattern,replace,string)

df['review'] = df['review'].apply(remove_url)

## 3 remove punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Za-z0-9\s  wanto keep it
    text = re.sub(r'@\w+', '', text)   # remove mentions
    text = re.sub(r'#\w+', '', text)   # remove hashtags
    text = re.sub(r'http\S+', '', text)

    return text

df['review'] = df['review'].apply(remove_punctuations)

## 4 remove html
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df['review'] = df['review'].apply(remove_html)


In [50]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [51]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [52]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")

  return text


df['review'] = df['review'].apply(remove_stopwords)

df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,review
0,0,3,0,0,3,2,rt myolovely womn nt complin b clening r ...
1,1,3,0,3,0,1,rt mleew17 boy dats coldtyga dwn bad cuff da...
2,2,3,0,3,0,1,rt urkindofbrnd dwg rt 80sbby4life ever fuck...
3,3,3,0,2,1,1,rt cgnderson vivbsed look like trnny
4,4,6,0,6,0,1,rt shenikarts sh hear might true might...


In [53]:
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,review
0,0,3,0,0,3,2,rt myolovely womn nt complin b clening r ...
1,1,3,0,3,0,1,rt mleew17 boy dats coldtyga dwn bad cuff da...
2,2,3,0,3,0,1,rt urkindofbrnd dwg rt 80sbby4life ever fuck...
3,3,3,0,2,1,1,rt cgnderson vivbsed look like trnny
4,4,6,0,6,0,1,rt shenikarts sh hear might true might...


In [54]:
#8 vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)

X=tf.fit_transform(df["review"])


In [55]:
X
y = df['class']

# datasets and dataloader

In [56]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [57]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

In [58]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 131085 stored elements and shape (19826, 5000)>

In [59]:
#Xtrain and test are sparse matrix so we convert to np.arrays
X_train = X_train.toarray()
X_test = X_test.toarray()

In [60]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).long()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).long()
)

In [61]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64,shuffle=True)

# build RNN

In [62]:
import torch.optim as optim

In [63]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
      super().__init__()

      self.hidden_size = hidden_size
      self.num_layers = num_layers

      #RNN LAYERS
      self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

      #fully connected layer
      self.fc =nn.Linear(hidden_size,3)

      # GIVING A MANY->ONE ARCHITECTURE

    def forward(self,x):
      #optional to write => shape(num of layers, batch size ,hidden size)
      h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

      out,_ = self.rnn(x,h0)
      #1st value = hidden state of all the timestamp => (batch,seq_len,hidden_Size)
      #2nd value = final hidden state of timestamp

      out = self.fc(out[:,-1,:])
      return out


In [64]:
input_size=X_train.shape[1]
model = RNN(input_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


# training the model

In [65]:
epochs =10

for epoch in range(epochs):
  model.train()

  for Xb,yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1) #the RNN model need a 3d data but data is 2d so it add a singleton dimension

    outputs = model(Xb) #return a 2d(batch_size,1)

    #outputs=torch.sigmoid(outputs.squeeze()) #tomake 1D we squezzed it

    loss=criterion(outputs,yb)#loss comput
    loss.backward()
    optimizer.step()


  print(f"Epoch: {epoch+1}/{epochs}, Loss: {loss.item()}")


Epoch: 1/10, Loss: 0.38532641530036926
Epoch: 2/10, Loss: 0.1459418386220932
Epoch: 3/10, Loss: 0.17156772315502167
Epoch: 4/10, Loss: 0.15049032866954803
Epoch: 5/10, Loss: 0.18639926612377167
Epoch: 6/10, Loss: 0.1322202831506729
Epoch: 7/10, Loss: 0.16779866814613342
Epoch: 8/10, Loss: 0.1766086220741272
Epoch: 9/10, Loss: 0.10342232882976532
Epoch: 10/10, Loss: 0.04262524098157883


In [66]:
model.eval()

with torch.no_grad():
  correct =0
  total=0

  for Xb,yb in test_loader:
    Xb = Xb.unsqueeze(1)
    outputs = model(Xb)
    predicted = torch.argmax(outputs,dim=1)

    correct+=(predicted==yb).sum().item()
    total+=yb.size(0)

  print(f"accuracy = {correct/total}")


accuracy = 0.8414363526326407


### Classifying a New Text Input

In [67]:
def preprocess_text(text):
    text = text.lower()
    text = remove_url(text)
    text = remove_punctuations(text)
    text = remove_html(text)
    text = remove_stopwords(text)
    return text

# Example text input
new_text = "this class is fucking amazing"

# Preprocess the new text
preprocessed_new_text = preprocess_text(new_text)
print(f"Original text: {new_text}")
print(f"Preprocessed text: {preprocessed_new_text}")

Original text: this class is fucking amazing
Preprocessed text:  class  fucking amazing


In [68]:
# Vectorize the preprocessed text using the trained TfidfVectorizer
# The tf object was already created and fitted on the training data.
text_vectorized = tf.transform([preprocessed_new_text])

# Convert to dense array if it's sparse
text_vectorized_array = text_vectorized.toarray()

print(f"Shape of vectorized text: {text_vectorized_array.shape}")

Shape of vectorized text: (1, 5000)


In [69]:
# Convert to PyTorch tensor
text_tensor = torch.from_numpy(text_vectorized_array).float()

# Add a sequence length dimension (RNN expects 3D input: batch, seq_len, features)
text_tensor = text_tensor.unsqueeze(1)

print(f"Shape of input tensor for RNN: {text_tensor.shape}")

Shape of input tensor for RNN: torch.Size([1, 1, 5000])


In [70]:
model.eval()

with torch.no_grad():
    output = model(text_tensor)              # (1, 3)
    prediction = torch.argmax(output, dim=1) # tensor([class])

# convert tensor → int
prediction = prediction.item()

# map classes
if prediction == 0:
    predicted_sentiment = "hatespeech"
elif prediction == 1:
    predicted_sentiment = "offensive"
else:
    predicted_sentiment = "neutral"

print(f"Predicted class: {prediction}")
print(f"Predicted sentiment: {predicted_sentiment}")

Predicted class: 0
Predicted sentiment: hatespeech


In [71]:
import pickle

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tf, f)

In [72]:
import torch

torch.save({
    "model_state": model.state_dict(),
    "input_size": X_train.shape[1]
}, "model.pth")

In [73]:
from google.colab import files

files.download("tfidf.pkl")
files.download("model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>